In [2]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils, drawing_styles

MODEL_PATH = "pose_landmarker.task"

base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    min_pose_detection_confidence=0.1,
    min_pose_presence_confidence=0.1,
    min_tracking_confidence=0.1
)
detector = vision.PoseLandmarker.create_from_options(options)



I0000 00:00:1775431765.934759   91141 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1775431765.974178   91142 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775431766.028296   91142 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [3]:
models = [
    "qwen2.5vl:7b",
    'gemma3:12b',
    "gemma4:e4b",
    #"qwen3-vl:8b",
    #"qwen2.5vl:32b",
    #"qwen3-vl:30b",
    #"llama3.2-vision:11b",
    "llava:7b"]
    

In [ ]:
ids=[0,2,3,4,5,6,7,8]
actuals=[8.5,7,7,6,8.5,6,7]

In [4]:
from utils import *

In [5]:
#pip install ollama
import ollama

In [6]:
from pydantic import BaseModel, Field

class ThrowCritique(BaseModel):
    score: int = Field(ge=1, le=10)
    feedback: str

In [7]:
class ThrowCritique(BaseModel):
    score: int = Field(ge=1, le=10)
    feedback: str = Field(
        min_length=20,
        max_length=300,
        description="Brief coaching feedback in 2-4 sentences."
    )

In [8]:
throwprompt2 = """
You are an elite Olympic discus coach and biomechanist.

You have been given a sequence of images showing a complete discus throw from beginning to end.

Evaluate the thrower's technique using these criteria:
- balance and posture
- rotation mechanics
- footwork
- hip and shoulder separation
- release position
- follow-through

Be strict but fair. Rate the throw from 1-10 where:
1 = very poor form
5 = average high school athlete
8 = strong collegiate athlete
10 = world class technique

Give 2-4 sentences of specific, actionable feedback.
"""

throwprompt3 = """
You are a discus coach whose job is to identify the single biggest flaw in a throw.

Look across the full sequence of images and determine:
1. The overall score from 1-10
2. The biggest mistake hurting the throw
3. The one change that would improve the throw the most

Keep the feedback concise and concrete. Do not mention uncertainty unless absolutely necessary.
"""

throwprompt4 = """
You are an AI sports judge trained to score discus technique from still frames.

You are seeing multiple frames sampled throughout one throw. Infer the movement between frames.

Score the throw from 1-10 using this rubric:
- 1-3: major balance, timing, or release problems
- 4-6: decent throw with noticeable flaws
- 7-8: technically good throw with minor issues
- 9-10: highly polished, competitive form

In your feedback, mention at least:
- one thing the athlete did well
- one thing to improve
- what phase of the throw looked weakest
"""

throwprompt5 = """
You are a brutally honest but helpful throwing coach.

Analyze the series of images as if you are watching a video of the throw. Pay close attention to:
- whether the athlete stays balanced
- whether the hips lead the shoulders
- whether the throwing arm lags correctly
- whether the release appears high and powerful

Give a score from 1-10. Then provide feedback in the style of a coach talking directly to the athlete: short, blunt, and practical.
"""

In [18]:

throwprompt="""
You are an expert AI discus throw coach.

You have been provided a series images of a pupil throwing a discus start to finish. 
The Images have a wire frame overlay to show posture.

Give your brief 3 bullet point feedback use only 2-4 sentences, you will be cut off after that.

"""

def critique_throw(image_paths: list,
                   mod: str = "llava:7b",
                   prompt: str = throwprompt,  
                   temp: float = 0.0):
    
    resp = ollama.chat(
        model=mod,
        messages=[
            {
                "role": "system",
                "content": prompt,
                "images": image_paths
            }
        ],
        format=ThrowCritique.model_json_schema(),
        options={"temperature": temp}
    )

    raw = resp["message"]["content"]
    return ThrowCritique.model_validate_json(raw)

In [10]:
prompts=[throwprompt,throwprompt2, throwprompt3,throwprompt4, throwprompt5]

In [11]:
import time
import pandas as pd

In [19]:
frames = sample_frames_from_mp4("videoplayback-2.mp4",10)
overs=pose_overlays_from_frames(frames,detector)
img_paths=write_temp_images_for_llm(overs)
crit=critique_throw(image_paths=img_paths,mod='gemma4:e4b',prompt=prompts[2])
crit

ThrowCritique(score=6, feedback="The biggest mistake is premature extension in the delivery. You are losing energy by 'throwing' with your arms instead of driving through the hips and core. Focus on keeping your chest ")

In [13]:
def doscore(count, instructions, mod):
    
    scores=[]
    durations=[]
    feedbacks=[]
    names=[]

    for i in range(1, 8):
        
        idstr = "" if i == 1 else f"-{i}"
        file = f"videoplayback{idstr}.mp4"
        start = time.time()

        try:
            frames = sample_frames_from_mp4(file, count)
            overs = pose_overlays_from_frames(frames, detector)
            img_paths = write_temp_images_for_llm(overs)
            crit = critique_throw(image_paths=img_paths, mod=mod, prompt=instructions)
            scores.append(crit.score)
            feedbacks.append(crit.feedback)
            names.append(file)
            
        except Exception as e:
            print(f"Failed on {file}: {e}")
            scores.append(pd.NA)
            feedbacks.append(pd.NA)
            names.append(file)

        durations.append(time.time() - start)

    return (
        pd.DataFrame({"score": scores, "duration": durations,
                      "feedback":feedbacks,"filenames":names}).\
            assign(frames=count, prompt=instructions[:20], model=mod)
    )

In [ ]:
report=pd.DataFrame()
for i in range(12,22,3):
    for j in prompts:
        for k in models:
            df=doscore(i,j,k)
            report=pd.concat([report,df])
            print(i,k)

3 qwen2.5vl:7b
3 gemma3:12b
3 llava:7b
3 qwen2.5vl:7b
3 gemma3:12b
3 llava:7b
3 qwen2.5vl:7b
3 gemma3:12b
3 llava:7b
3 qwen2.5vl:7b
3 gemma3:12b
3 llava:7b
3 qwen2.5vl:7b
3 gemma3:12b
3 llava:7b
6 qwen2.5vl:7b
6 gemma3:12b
6 llava:7b
6 qwen2.5vl:7b
Failed on videoplayback-3.mp4: Failed to create new sequence: failed to process inputs: png: invalid format: not enough pixel data (status code: 500)
6 gemma3:12b
6 llava:7b
6 qwen2.5vl:7b
6 gemma3:12b
6 llava:7b
6 qwen2.5vl:7b
6 gemma3:12b
6 llava:7b
6 qwen2.5vl:7b
6 gemma3:12b
6 llava:7b
9 qwen2.5vl:7b
9 gemma3:12b
9 llava:7b
9 qwen2.5vl:7b
9 gemma3:12b
9 llava:7b
9 qwen2.5vl:7b
9 gemma3:12b
9 llava:7b
9 qwen2.5vl:7b
9 gemma3:12b
9 llava:7b
9 qwen2.5vl:7b
9 gemma3:12b
9 llava:7b
12 qwen2.5vl:7b
12 gemma3:12b
12 llava:7b
12 qwen2.5vl:7b
12 gemma3:12b
12 llava:7b
12 qwen2.5vl:7b
12 gemma3:12b
12 llava:7b
12 qwen2.5vl:7b
12 gemma3:12b
12 llava:7b
12 qwen2.5vl:7b
12 gemma3:12b
12 llava:7b
Failed on videoplayback-4.mp4: an error was encountered

In [ ]:
report.to_csv("wannoreport2.csv")

In [ ]:
import os
import time

while True:
    os.system('say "I am done now"')
    time.sleep(2)  # wait 2 seconds before saying it again